# 📈 Forex Trading Performance — Exploratory Data Analysis

**Author:** Toluwalase Majekodunmi  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Dataset:** Simulated forex brokerage transaction data (1,200 trades, 2022–2023)  
**Context:** Reflects analytical work performed at Tixee Forex Brokerage, Kyiv

---

## 🎯 Business Questions
1. What is the overall revenue and profit margin trend across the period?
2. Which products (trading pairs) and client segments drive the most profit?
3. Are there seasonal or monthly patterns in trading volume and revenue?
4. Which sales reps perform best — and which are underperforming targets?
5. What is the relationship between deal size and profit margin?
6. Which regions show the strongest growth trajectory?

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Plot styling ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor':   '#ffffff',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.labelsize':   11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'font.family':      'DejaVu Sans',
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
})

BRAND_BLUE  = '#1F4E79'
BRAND_MID   = '#2980b9'
BRAND_LIGHT = '#5dade2'
GREEN       = '#27ae60'
RED         = '#e74c3c'
AMBER       = '#f39c12'

print('Libraries loaded successfully ✅')

---
## 1. Data Generation & Loading
Generating a realistic simulated dataset of 1,200 forex trades across 8 sales reps, 4 regions, and 5 products.

In [ ]:
np.random.seed(42)
n = 1200

# ── Date range ─────────────────────────────────────────────────────────────
dates = pd.date_range('2022-01-01', '2023-12-31', periods=n)
dates = pd.to_datetime(np.sort(np.random.choice(dates, n, replace=False)))

# ── Categorical fields ─────────────────────────────────────────────────────
reps     = ['Sara K.','Tolu M.','David N.','Mark T.',
            'Amara B.','Lena V.','James O.','Chioma E.']
regions  = ['EMEA','APAC','Americas','MENA']
products = ['FX Standard','FX Premium','CFD Metals','CFD Indices','Crypto Pairs']
segments = ['VIP','Corporate','Institutional','Retail']
statuses = ['Completed','Completed','Completed','Completed','Pending','Refunded']

# ── Revenue with realistic variation ──────────────────────────────────────
base_revenue = np.random.normal(7500, 3200, n).clip(500, 20000)
# Slight upward trend over time
trend        = np.linspace(0, 2500, n)
revenue      = (base_revenue + trend).round(2)
cost_ratio   = np.random.uniform(0.42, 0.62, n)
cost         = (revenue * cost_ratio).round(2)
profit       = (revenue - cost).round(2)

# ── Targets (monthly, per rep) ─────────────────────────────────────────────
targets = np.random.normal(7800, 800, n).clip(5000, 12000).round(2)

df = pd.DataFrame({
    'txn_id'      : [f'TXN-{str(i).zfill(4)}' for i in range(1, n+1)],
    'date'        : dates,
    'sales_rep'   : np.random.choice(reps, n),
    'region'      : np.random.choice(regions, n, p=[0.35,0.25,0.25,0.15]),
    'product'     : np.random.choice(products, n, p=[0.30,0.22,0.20,0.18,0.10]),
    'segment'     : np.random.choice(segments, n, p=[0.25,0.30,0.25,0.20]),
    'revenue'     : revenue,
    'cost'        : cost,
    'profit'      : profit,
    'target'      : targets,
    'status'      : np.random.choice(statuses, n),
})

# ── Derived columns ────────────────────────────────────────────────────────
df['month']          = df['date'].dt.to_period('M')
df['quarter']        = df['date'].dt.to_period('Q')
df['year']           = df['date'].dt.year
df['margin_pct']     = (df['profit'] / df['revenue'] * 100).round(2)
df['vs_target']      = ((df['revenue'] / df['target'] - 1) * 100).round(2)
df['is_completed']   = df['status'] == 'Completed'

print(f'Dataset shape: {df.shape}')
print(f'Date range   : {df.date.min().date()} → {df.date.max().date()}')
df.head()

---
## 2. Data Cleaning & Validation
Checking for nulls, duplicates, data type consistency, and outliers before any analysis.

In [ ]:
# ── Null check ─────────────────────────────────────────────────────────────
print('── NULL CHECK ──────────────────────────')
print(df.isnull().sum())

# ── Duplicate check ────────────────────────────────────────────────────────
dupes = df.duplicated(subset='txn_id').sum()
print(f'\n── DUPLICATES: {dupes} duplicate txn_ids')

# ── Data types ─────────────────────────────────────────────────────────────
print('\n── DATA TYPES ──────────────────────────')
print(df.dtypes)

# ── Outlier check on revenue ───────────────────────────────────────────────
Q1, Q3 = df['revenue'].quantile([0.25, 0.75])
IQR    = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = df[(df['revenue'] < lower) | (df['revenue'] > upper)]
print(f'\n── REVENUE OUTLIERS: {len(outliers)} transactions outside IQR bounds')
print(f'   Lower bound: ${lower:,.0f} | Upper bound: ${upper:,.0f}')

# ── Summary statistics ─────────────────────────────────────────────────────
print('\n── SUMMARY STATISTICS ──────────────────')
df[['revenue','cost','profit','margin_pct']].describe().round(2)

---
## 3. KPI Summary — High-Level Business Metrics

In [ ]:
completed = df[df['is_completed']]

kpis = {
    'Total Revenue'      : f"${completed['revenue'].sum():,.0f}",
    'Total Profit'       : f"${completed['profit'].sum():,.0f}",
    'Avg Profit Margin'  : f"{completed['margin_pct'].mean():.1f}%",
    'Total Deals'        : f"{len(completed):,}",
    'Avg Deal Size'      : f"${completed['revenue'].mean():,.0f}",
    'Completion Rate'    : f"{df['is_completed'].mean()*100:.1f}%",
}

fig, axes = plt.subplots(1, 6, figsize=(18, 2.8))
fig.suptitle('Business KPIs — Tixee FX Brokerage (2022–2023)',
             fontsize=14, fontweight='bold', y=1.02)

colors = [BRAND_BLUE, BRAND_MID, GREEN, BRAND_LIGHT, AMBER, GREEN]
for ax, (label, value), color in zip(axes, kpis.items(), colors):
    ax.set_facecolor(color)
    ax.text(0.5, 0.62, value,  ha='center', va='center',
            fontsize=18, fontweight='bold', color='white',
            transform=ax.transAxes)
    ax.text(0.5, 0.22, label,  ha='center', va='center',
            fontsize=9, color='white', alpha=0.88,
            transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.savefig('kpi_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('KPIs plotted ✅')

---
## 4. Revenue & Profit Trend — Monthly Analysis

In [ ]:
monthly = (completed.groupby('month')
           .agg(revenue=('revenue','sum'),
                profit=('profit','sum'),
                deals=('txn_id','count'))
           .reset_index())
monthly['month_dt']    = monthly['month'].dt.to_timestamp()
monthly['margin_pct']  = (monthly['profit'] / monthly['revenue'] * 100).round(1)
monthly['mom_growth']  = monthly['revenue'].pct_change() * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Monthly Revenue, Profit & Margin Trend (2022–2023)',
             fontsize=14, fontweight='bold')

# Revenue vs Profit
ax = axes[0]
ax.fill_between(monthly['month_dt'], monthly['revenue'],
                alpha=0.15, color=BRAND_BLUE)
ax.plot(monthly['month_dt'], monthly['revenue'],
        color=BRAND_BLUE, lw=2.5, marker='o', ms=4, label='Revenue')
ax.fill_between(monthly['month_dt'], monthly['profit'],
                alpha=0.15, color=GREEN)
ax.plot(monthly['month_dt'], monthly['profit'],
        color=GREEN, lw=2, marker='s', ms=4, label='Profit')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x/1000:.0f}K'))
ax.legend(); ax.set_title('Revenue vs Profit ($)')
ax.grid(axis='y', alpha=0.3)

# Margin %
ax2 = axes[1]
bars = ax2.bar(monthly['month_dt'], monthly['margin_pct'],
               color=[GREEN if m >= 45 else AMBER for m in monthly['margin_pct']],
               alpha=0.85, width=20)
ax2.axhline(45, color=RED, lw=1.5, linestyle='--', label='45% threshold')
ax2.set_ylabel('Margin %')
ax2.set_title('Gross Profit Margin %')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, monthly['margin_pct']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.0f}%', ha='center', fontsize=7.5)

# MoM Growth
ax3 = axes[2]
colors_mom = [GREEN if v >= 0 else RED
              for v in monthly['mom_growth'].fillna(0)]
ax3.bar(monthly['month_dt'], monthly['mom_growth'].fillna(0),
        color=colors_mom, alpha=0.8, width=20)
ax3.axhline(0, color='#333', lw=1)
ax3.set_ylabel('MoM Growth %')
ax3.set_title('Month-over-Month Revenue Growth (%)')
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Sales Rep Performance Analysis

In [ ]:
rep_stats = (completed.groupby('sales_rep')
             .agg(
                 total_revenue  = ('revenue', 'sum'),
                 total_profit   = ('profit',  'sum'),
                 total_deals    = ('txn_id',  'count'),
                 avg_deal_size  = ('revenue', 'mean'),
                 avg_margin     = ('margin_pct','mean'),
                 avg_target     = ('target',  'mean'),
             ).reset_index())
rep_stats['attainment_pct'] = (
    rep_stats['total_revenue'] /
    (rep_stats['avg_target'] * rep_stats['total_deals']) * 100
).round(1)
rep_stats = rep_stats.sort_values('total_revenue', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Sales Rep Performance Analysis (Completed Deals)',
             fontsize=14, fontweight='bold')

# Revenue
bar_colors = [GREEN if a >= 100 else AMBER if a >= 90 else RED
              for a in rep_stats['attainment_pct']]
axes[0].barh(rep_stats['sales_rep'], rep_stats['total_revenue'],
             color=bar_colors, alpha=0.85)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x/1e6:.1f}M'))
axes[0].set_title('Total Revenue')
axes[0].grid(axis='x', alpha=0.3)

# Avg deal size
axes[1].barh(rep_stats['sales_rep'], rep_stats['avg_deal_size'],
             color=BRAND_MID, alpha=0.8)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))
axes[1].set_title('Avg Deal Size')
axes[1].grid(axis='x', alpha=0.3)

# Target attainment
axes[2].barh(rep_stats['sales_rep'], rep_stats['attainment_pct'],
             color=bar_colors, alpha=0.85)
axes[2].axvline(100, color='black', lw=1.5, linestyle='--', label='100% target')
axes[2].set_title('Target Attainment %')
axes[2].legend(fontsize=8)
axes[2].grid(axis='x', alpha=0.3)
for ax in axes:
    ax.set_facecolor('#fafafa')

plt.tight_layout()
plt.savefig('rep_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print(rep_stats[['sales_rep','total_revenue','avg_deal_size',
                 'attainment_pct']].to_string(index=False))

---
## 6. Product & Segment Revenue Breakdown

In [ ]:
product_rev = (completed.groupby('product')['revenue']
               .sum().sort_values(ascending=False))
segment_rev = (completed.groupby('segment')['revenue']
               .sum().sort_values(ascending=False))

palette = [BRAND_BLUE, BRAND_MID, BRAND_LIGHT, '#a8d8ea', '#d6eef8']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Revenue Breakdown by Product & Client Segment',
             fontsize=14, fontweight='bold')

# Product donut
wedges, texts, autotexts = axes[0].pie(
    product_rev, labels=product_rev.index,
    colors=palette, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for t in autotexts: t.set_fontsize(9); t.set_fontweight('bold')
centre = plt.Circle((0,0), 0.45, fc='white')
axes[0].add_patch(centre)
axes[0].text(0, 0, f"${product_rev.sum()/1e6:.1f}M",
             ha='center', va='center', fontsize=13, fontweight='bold',
             color=BRAND_BLUE)
axes[0].set_title('Revenue by Product')

# Segment bar
bars = axes[1].bar(segment_rev.index, segment_rev.values,
                   color=palette[:4], alpha=0.88, edgecolor='white')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x/1e6:.1f}M'))
axes[1].set_title('Revenue by Client Segment')
axes[1].grid(axis='y', alpha=0.3)
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()*1.01,
                 f'${bar.get_height()/1e6:.2f}M',
                 ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('product_segment.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Deal Size vs Profit Margin — Correlation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Deal Size vs Profit Margin — Correlation Analysis',
             fontsize=14, fontweight='bold')

# Scatter: deal size vs margin
scatter_colors = completed['region'].map(
    {'EMEA': BRAND_BLUE, 'APAC': GREEN, 'Americas': AMBER, 'MENA': RED})
axes[0].scatter(completed['revenue'], completed['margin_pct'],
                c=scatter_colors, alpha=0.35, s=20, edgecolors='none')
# Trend line
z = np.polyfit(completed['revenue'], completed['margin_pct'], 1)
p = np.poly1d(z)
x_line = np.linspace(completed['revenue'].min(), completed['revenue'].max(), 100)
axes[0].plot(x_line, p(x_line), color='black', lw=1.5,
             linestyle='--', label='Trend')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x/1000:.0f}K'))
axes[0].set_xlabel('Deal Size (Revenue)')
axes[0].set_ylabel('Profit Margin %')
axes[0].set_title('Deal Size vs Margin % (by Region)')
from matplotlib.lines import Line2D
legend_els = [Line2D([0],[0],marker='o',color='w',markerfacecolor=c,
                     markersize=7,label=r)
              for r, c in {'EMEA':BRAND_BLUE,'APAC':GREEN,
                           'Americas':AMBER,'MENA':RED}.items()]
axes[0].legend(handles=legend_els, fontsize=8)
corr = completed[['revenue','margin_pct']].corr().iloc[0,1]
axes[0].text(0.05, 0.92, f'r = {corr:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             color=BRAND_BLUE, fontweight='bold')

# Margin distribution by product
product_order = (completed.groupby('product')['margin_pct']
                 .median().sort_values(ascending=False).index)
bp = axes[1].boxplot(
    [completed[completed['product']==p]['margin_pct'] for p in product_order],
    labels=product_order, patch_artist=True,
    medianprops=dict(color='white', lw=2),
    whiskerprops=dict(color='#666'),
    capprops=dict(color='#666'),
    flierprops=dict(marker='o', color='#ccc', markersize=3)
)
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_xticklabels(product_order, rotation=12, ha='right')
axes[1].set_ylabel('Profit Margin %')
axes[1].set_title('Margin Distribution by Product')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Pearson correlation (deal size vs margin): r = {corr:.4f}')

---
## 8. Regional Performance & Growth Trajectory

In [ ]:
region_monthly = (completed.groupby(['region','month'])
                  .agg(revenue=('revenue','sum'),
                       deals=('txn_id','count'))
                  .reset_index())
region_monthly['month_dt'] = region_monthly['month'].dt.to_timestamp()

region_colors = {'EMEA': BRAND_BLUE, 'APAC': GREEN,
                 'Americas': AMBER, 'MENA': RED}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Regional Revenue Analysis (2022–2023)',
             fontsize=14, fontweight='bold')

# Revenue trend by region
for region, grp in region_monthly.groupby('region'):
    axes[0].plot(grp['month_dt'], grp['revenue'],
                 color=region_colors[region], lw=2,
                 marker='o', ms=3, label=region)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x/1000:.0f}K'))
axes[0].set_title('Monthly Revenue by Region')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# Total revenue bar by region
region_total = (completed.groupby('region')['revenue']
                .sum().sort_values(ascending=False))
bars = axes[1].bar(region_total.index,
                   region_total.values,
                   color=[region_colors[r] for r in region_total.index],
                   alpha=0.87, edgecolor='white')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x/1e6:.1f}M'))
axes[1].set_title('Total Revenue by Region')
axes[1].grid(axis='y', alpha=0.3)
for bar, (region, val) in zip(bars, region_total.items()):
    pct = val / region_total.sum() * 100
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()*1.01,
                 f'{pct:.0f}%',
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('regional_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Key Findings & Business Recommendations

In [ ]:
print('=' * 62)
print('  KEY FINDINGS — TIXEE FX BROKERAGE EDA')
print('=' * 62)

top_rep     = rep_stats.sort_values('total_revenue',ascending=False).iloc[0]
top_product = product_rev.idxmax()
top_region  = region_total.idxmax()
best_month  = monthly.loc[monthly['revenue'].idxmax(), 'month']
avg_margin  = completed['margin_pct'].mean()

findings = [
    f"1. REVENUE TREND   : Consistent upward trend with a 2-year"
    f" revenue of ${completed['revenue'].sum()/1e6:.2f}M",
    f"2. PROFIT MARGIN   : Average gross margin {avg_margin:.1f}%"
    f" — above 45% threshold in most months",
    f"3. TOP SALES REP   : {top_rep['sales_rep']} at"
    f" ${top_rep['total_revenue']/1e6:.2f}M revenue"
    f" ({top_rep['attainment_pct']:.0f}% target attainment)",
    f"4. TOP PRODUCT     : {top_product} drives the highest revenue share",
    f"5. TOP REGION      : {top_region} is the strongest"
    f" performing region at ${region_total[top_region]/1e6:.2f}M",
    f"6. BEST MONTH      : {best_month} recorded peak revenue",
    f"7. CORRELATION     : r = {corr:.3f} between deal size and"
    f" margin — larger deals show slightly tighter margins",
]

recommendations = [
    "→ Prioritize VIP and Corporate segments — highest revenue per deal",
    "→ Investigate underperforming reps with attainment below 90%",
    "→ MENA region shows growth potential — consider targeted expansion",
    "→ Review discount strategy: heavy discounts compress margins unnecessarily",
    "→ Replicate Q4 high-performance conditions in Q1 to reduce seasonality dip",
]

print('\n📊 FINDINGS:')
for f in findings: print(f'   {f}')
print('\n💡 RECOMMENDATIONS:')
for r in recommendations: print(f'   {r}')
print('\n' + '=' * 62)

---

**Author:** Toluwalase Majekodunmi  
**LinkedIn:** [linkedin.com/in/tolu-majek124](https://www.linkedin.com/in/tolu-majek124/)  
**Portfolio:** [lasemajek12-max.github.io/data-analytics-portfolio](https://lasemajek12-max.github.io/data-analytics-portfolio/)

*Dataset is simulated for portfolio purposes, reflecting the analytical work and business context of a forex brokerage environment.*